# Figure generation

Run top to bottom. Writes PNG (600 dpi, for Word) to `png/` and vector PDF
to `pdf/`. Cell 4 (process performance) reads
`../Experiments/BCAT1_KO Fed-Batch.xlsx`, so run with this folder's parent
as the working directory.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

OUT = Path.cwd()          # write beside this notebook

def save(fig, name):
    fig.savefig(OUT / f'{name}.pdf', bbox_inches='tight')
    fig.savefig(OUT / f'{name}.png', dpi=600, bbox_inches='tight')
    print('wrote', name)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np
from matplotlib.patches import FancyBboxPatch

# === DATA ===
scenarios = [
    # D1 scenarios
    (1, 'D1', 'Bio_Igg', 0.0066, 21.2311, 8),
    (3, 'D1', 'Bio_Igg', 0.0057, 24.7229, 4),
    (9, 'D1', 'Igg_Igg', 0.0046, 28.9245, 11),
    (12, 'D1', 'Igg_Igg', 0.0101, 8.4772, 5),
    (27, 'D1', 'Igg_Igg', 0.0055, 25.4711, 9),
    (47, 'D1', 'Igg_Igg', 0.0087, 13.4181, 2),
    (80, 'D1', 'Bio_Igg', 0.0045, 29.0373, 4),
    (81, 'D1', 'Bio_Igg', 0.0062, 22.9280, 9),
    (84, 'D1', 'Bio_Igg', 0.0066, 21.4020, 8),
    (86, 'D1', 'Bio_Igg', 0.0055, 25.6321, 5),
    (88, 'D1', 'Bio_Igg', 0.0085, 14.3019, 20),
    # D2 scenarios
    (30, 'D2', 'Igg_Igg', 0.0099, 63.6437, 13),
    (31, 'D2', 'Igg_Igg', 0.0173, 46.0704, 2),
    (74, 'D2', 'Igg_Igg', 0.0111, 32.7011, 10),
    # D3 scenarios
    (32, 'D3', 'Bio_Igg', 0.0205, 34.0823, 2),
    (33, 'D3', 'Bio_Igg', 0.0205, 34.0823, 2),
    (47, 'D3', 'Igg_Igg', 0.0144, 30.115, 8),
    # D4 scenarios
    (14, 'D4', 'Bio_Igg', 0.0133, 61.0383, 6),
    (49, 'D4', 'Bio_Igg', 0.0130, 39.3633, 7),
    (52, 'D4', 'Bio_Igg', 0.0124, 43.6783, 7),
    (58, 'D4', 'Bio_Igg', 0.0134, 60.7551, 2),
]

controls = {
    'D1': (0.0088, 16.2),
    'D2': (0.0188, 48.0),
    'D3': (0.0223, 31.7),
    'D4': (0.0215, 51.8),
}

ds_markers = {'D1': 'o', 'D2': 's', 'D3': '^', 'D4': 'D'}

# Colour map
all_regs = [s[5] for s in scenarios]
cmap = cm.YlOrRd
norm = mcolors.Normalize(vmin=min(all_regs) - 0.5, vmax=max(all_regs) + 0.5)

# === FIGURE ===
fig = plt.figure(figsize=(16.5, 6.4))
gs = fig.add_gridspec(1, 3, width_ratios=[0.25, 0.27, 0.48], wspace=0.20)

# --- PANEL A: PREPROCESSING ---
ax_prep = fig.add_subplot(gs[0])
ax_prep.set_xlim(0, 10)
ax_prep.set_ylim(-0.5, 13.8)
ax_prep.axis('off')

# --- PANEL B: SCENARIO REDUCTION ---
ax_flow = fig.add_subplot(gs[1])
ax_flow.set_xlim(0, 10)
ax_flow.set_ylim(-0.5, 10)
ax_flow.axis('off')

def add_box(ax, x, y, width, height, title, lines, color,
            fs_title=11, fs_line=9.5, step=0.3):
    box = FancyBboxPatch((x, y), width, height,
                          boxstyle="round,pad=0.08",
                          facecolor=color,
                          edgecolor='#666666',
                          linewidth=1.5,
                          alpha=0.95)
    ax.add_patch(box)
    ax.text(x + width/2, y + height - 0.25, title,
            ha='center', va='top', fontsize=fs_title, fontweight='bold')
    y_text = y + height - 0.7
    for line in lines:
        ax.text(x + width/2, y_text, line,
                ha='center', va='top', fontsize=fs_line)
        y_text -= step

def add_arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', lw=2.5, color='#666666'))
    if label:
        label_lines = label.split('\n')
        y_offset = (y1 + y2)/2
        for i, line in enumerate(label_lines):
            ax.text(5.8, y_offset - i*0.28, line,
                    fontsize=8.5, ha='left', style='italic', color='#424242')

prep_steps = [
    ('STARTING GEM',
     ['iCHO1766-K1 (Hefzi et al., 2016)',
      '3952 reactions, 2026 metabolites',
      '1202 genes with GPR rules']),
    ('REMOVE UNNECESSARY REACTIONS',
     ['heterologous product synthesis',
      'not expressed by the host line']),
    ('CONDITION-SPECIFIC BOUNDS',
     ['measured uptake and secretion rates',
      'four feeding regimes (D1–D4)']),
    ('RESTRICT THE SEARCH SPACE',
     ['essential 211–236, multi-gene 337,',
      'exchange 420, blocked 693',
      '1354–1379 reactions excluded']),
    ('CONVERT TO IRREVERSIBLE',
     ['COBRA convertToIrreversible',
      '3952 → 4940 unidirectional',
      'flux variables']),
]

_pb_h = 2.0
_pb_gap = 0.78
_pb_top = 13.3
for _i, (_t, _ls) in enumerate(prep_steps):
    _y = _pb_top - _i * (_pb_h + _pb_gap) - _pb_h
    add_box(ax_prep, 0.3, _y, 9.4, _pb_h, _t, _ls, '#EDE7F6',
            fs_title=9.5, fs_line=8.3)
    if _i < len(prep_steps) - 1:
        add_arrow(ax_prep, 5, _y - 0.05, 5, _y - _pb_gap + 0.05)

ax_prep.text(-0.5, 13.8, 'A', fontsize=18, fontweight='bold')

add_box(ax_flow, 0.3, 7.2, 9.4, 2.5,
        'BILEVEL OPTIMISATION',
        ['1,300+ optimisation runs',
         '\u2192 93 unique scenarios',
         '',
         'Bio_Igg: 53  |  Igg_Igg: 40',
         'Average: 130 regulations/scenario'],
        '#D6EAF8')

add_arrow(ax_flow, 5, 7.1, 5, 6.3, 'Sensitivity\nAnalysis')

add_box(ax_flow, 0.3, 3.6, 9.4, 2.5,
        'REGULATION REDUCTION',
        ['93 \u2192 89 scenarios',
         '(4 duplicates removed)',
         '',
         '130 \u2192 9 regulations (avg)',
         '\u2022 All downregulations removed'],
        '#D5F4E6')

add_arrow(ax_flow, 5, 3.5, 5, 2.7, '\u03bc >50% AND\nq$_{mAb}$ >50%')

add_box(ax_flow, 0.3, 0.0, 9.4, 2.5,
        'HIGH-PRODUCING SCENARIOS',
        ['89 \u2192 20 scenarios (22%)',
         '',
         'Average: 6.8 regulations',
         'Range: 2\u201320 regulations',
         '\u2192 Ready for analysis'],
        '#FCE8D5')

ax_flow.text(-0.5, 10, 'B', fontsize=18, fontweight='bold')

# --- PANEL C: SCENARIO LANDSCAPE ---
ax = fig.add_subplot(gs[2])

# Plot control points (hollow)
for ds, (mu, qmab) in controls.items():
    ax.scatter(mu, qmab, marker=ds_markers[ds], s=150,
              facecolors='none', edgecolors='black', linewidths=2.0, zorder=4)

# Label controls
ctrl_offsets = {
    'D1': (0.0012, -1.0),
    'D2': (0.0008, 1.8),
    'D3': (0.0008, -3.5),
    'D4': (0.0005, 2.0),
}
for ds, (mu, qmab) in controls.items():
    dx, dy = ctrl_offsets[ds]
    ax.text(mu + dx, qmab + dy, ds, fontsize=9, fontstyle='italic',
            fontweight='bold', color='#333333', ha='left', va='bottom')

# Plot all scenarios including 52 with normal markers
s52 = [s for s in scenarios if s[0] == 52][0]
for scen, ds, algo, mu, qmab, nregs in scenarios:
    colour = cmap(norm(nregs))
    ax.scatter(mu, qmab, marker=ds_markers[ds], s=85,
              c=[colour], edgecolors='#444444', linewidths=0.7,
              zorder=3, alpha=0.85)

# Circle highlight around Scenario 52
from matplotlib.patches import Ellipse
# Use ellipse to account for different axis scales
x_range = 0.025 - 0.002
y_range = 68 - 5
aspect = (x_range / y_range)  # ratio of axis ranges
circle_r_y = 3.5  # radius in y-data units
circle_r_x = circle_r_y * aspect * (6/8)  # adjust for figure aspect ratio
ellipse = Ellipse((s52[3], s52[4]), width=2*circle_r_x, height=2*circle_r_y,
                  fill=False, edgecolor='black', linewidth=1.8, linestyle='-', zorder=5)
ax.add_patch(ellipse)
ax.annotate('Scenario 52',
           xy=(s52[3], s52[4] + circle_r_y),
           xytext=(0.006, 57),
           bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                    edgecolor='black', linewidth=1.2),
           arrowprops=dict(arrowstyle='->', lw=1.5, color='black'),
           fontsize=9, fontweight='bold', ha='center', zorder=6)

# Legend
ax.scatter([], [], marker='o', s=120, facecolors='none', edgecolors='black',
          linewidths=2, label='FBA control')
for ds, marker in ds_markers.items():
    ax.scatter([], [], marker=marker, s=70, c='#D4A574', edgecolors='#444444',
              linewidths=0.7, label=ds)

ax.legend(loc='lower right', frameon=True, fancybox=True,
         fontsize=9, framealpha=0.95, edgecolor='gray')

# Colourbar
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.02, shrink=0.85)
cbar.set_label('Number of regulations\n(post-sensitivity analysis)', fontsize=10)
cbar.set_ticks([2, 5, 8, 11, 14, 17, 20])

ax.set_xlabel('Growth rate, \u03bc (hr$^{-1}$)', fontsize=12)
ax.set_ylabel('Specific productivity, q$_{mAb}$ (pg/cell/day)', fontsize=12)

ax.grid(True, alpha=0.2, linestyle=':', linewidth=0.7)
ax.set_xlim(0.002, 0.025)
ax.set_ylim(5, 68)

ax.text(-0.13, 1.06, 'C', transform=ax.transAxes,
       fontsize=18, fontweight='bold')

plt.tight_layout()
save(plt.gcf(), 'Fig1_Model preparation, scenario selection and the resulting design space')
print("Done")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ── Publication style ────────────────────────────────────────────────
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
})

# ── Data ─────────────────────────────────────────────────────────────
intervals = ["5–6", "6–7", "7–8", "8–9", "9–10", "10–11",
             "11–12", "12–13", "13–14"]

# CHO' (process control) — df1
flux_CHOp = np.array([-0.0051, -0.0111, -0.0014, 0.0008, -0.0070,
                      -0.0099, 0.0025, -0.0146, -0.0027])
err_CHOp  = np.array([0.0069, 0.0020, 0.0020, 0.0045, 0.0081,
                      0.0049, 0.0008, 0.0015, 0.0022])

# CHO-B (engineered) — df2
flux_CHOB = np.array([-0.0002, -0.0148, -0.0006, -0.0092, -0.0031,
                      -0.0101, -0.0041, -0.0092, -0.0018])
err_CHOB  = np.array([0.0063, 0.0018, 0.0006, 0.0006, 0.0024,
                      0.0012, 0.0021, 0.0031, 0.0017])

# CHO (parental) — df3
flux_CHO = np.array([-0.0025, -0.0101, -0.0021, -0.0068, -0.0009,
                     -0.0122, -0.0018, -0.0109, -0.0031])
err_CHO  = np.array([0.0014, 0.0052, 0.0017, 0.0038, 0.0017,
                     0.0050, 0.0012, 0.0074, 0.0036])

x = np.arange(len(intervals))

# ── Colours ──────────────────────────────────────────────────────────
c_CHO  = '#2166AC'   # blue
c_CHOp = '#E08214'   # orange
c_CHOB = '#1A9850'   # green

# ── Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))

# Lines with error bars (no markers on mean line, caps on error bars)
ax.errorbar(x, flux_CHO, yerr=err_CHO, color=c_CHO, linewidth=1.2,
            capsize=3, capthick=0.8, elinewidth=0.8, label='CHO')

ax.errorbar(x, flux_CHOp, yerr=err_CHOp, color=c_CHOp, linewidth=1.2,
            capsize=3, capthick=0.8, elinewidth=0.8, label="CHO\u2032")

ax.errorbar(x, flux_CHOB, yerr=err_CHOB, color=c_CHOB, linewidth=1.2,
            capsize=3, capthick=0.8, elinewidth=0.8, label='CHO-B')

# Zero line
ax.axhline(0, color='0.6', linewidth=0.5, linestyle='-', zorder=1)

# Axes
ax.set_xticks(x)
ax.set_xticklabels(intervals, rotation=45, ha='right')
ax.set_xlabel('Culture interval (days)')
ax.set_ylabel('Specific leucine uptake rate (mmol/gDCW/hr)')

ax.legend(loc='lower left')
plt.tight_layout()

# superseded working figure, no longer written
plt.show()


In [ ]:
"""
Publication-quality figures for CHO BCAT1 knockdown paper.
Generates:
  - Figure 5: Growth profiles (VCD over 14 days)
  - Figure 6: Antibody production (titre + specific productivity)

All figures use consistent styling and colour scheme:
  CHO  = blue (#2166AC)
  CHO' = orange (#E08214)
  CHO-B = green (#1A9850)
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import FancyBboxPatch

# ══════════════════════════════════════════════════════════════════════
# GLOBAL STYLE
# ══════════════════════════════════════════════════════════════════════
mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.major.size': 4,
    'ytick.major.size': 4,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'legend.frameon': False,
})

c_CHO  = '#2166AC'
c_CHOp = '#E08214'
c_CHOB = '#1A9850'

# ══════════════════════════════════════════════════════════════════════
# DATA (from Excel file s.xlsx)
# ══════════════════════════════════════════════════════════════════════
days = np.arange(0, 15)

# --- VCD (1E6 cells/mL) ---
vcd_CHOp = np.array([0.2, 0.52, 1.12, 1.15, 3.21, 6.608, 8.0,
                     10.525, 10.725, 10.475, 10.225, 10.275, 8.45,
                     8.025, 5.6])
vcd_CHOp_err = np.array([0, 0.0226, 0.2263, 0.0707, 0.0141, 0.1584,
                         1.1314, 0.3182, 0.3182, 0.3889, 0.3182,
                         1.0253, 0.9192, 1.0960, 0.6223])

vcd_CHO = np.array([0.2, 0.378, 0.793, 1.48, 4.872, 4.95, 6.14,
                    8.775, 9.75, 10.85, 10.55, 8.7, 7.35, 6.9, 6.66])
vcd_CHO_err = np.array([0, 0.0594, 0.0240, 0.3111, 1.5952, 1.4001,
                        0.9334, 1.8031, 0.7071, 1.2021, 0.1414,
                        0.7778, 0.2121, 0.1414, 0.3677])

vcd_CHOB = np.array([0.2, 0.396, 0.68, 0.935, 2.856, 4.122, 5.712,
                     7.76, 9.8, 9.78, 9.02, 6.88, 8.34, 6.8, 5.68])
vcd_CHOB_err = np.array([0, 0, 0.0905, 0.0495, 0.4752, 0.2461,
                         1.1993, 1.0748, 0.7920, 0.8202, 1.7253,
                         0.1131, 0.1980, 1.2445, 0.1131])

# --- Titre (g/L) ---
titre_CHOp = np.array([0.00447, 0.01906, 0.04229, 0.07261, 0.21925,
                       0.36605, 0.58195, 0.79365, 1.257, 1.5045,
                       1.8145, 1.9945, 1.8795, 1.8545, 1.783])
titre_CHOp_err = np.array([4.95e-5, 5.73e-4, 5.04e-3, 0.01563, 3.18e-3,
                           0.02270, 0.12735, 0.03599, 0.19940, 0.11526,
                           0.08980, 0.04172, 7.07e-4, 0.05728, 0.02687])

titre_CHO = np.array([0.00302, 0.01739, 0.03257, 0.0643, 0.1552,
                      0.24705, 0.45015, 0.60885, 0.72875, 0.93445,
                      1.285, 1.695, 1.875, 1.928, 1.844])
titre_CHO_err = np.array([7.33e-4, 2.26e-4, 3.75e-3, 5.94e-4, 2.12e-3,
                          0.01082, 4.17e-3, 0.05819, 0.03288, 0.01761,
                          0.05798, 8.49e-3, 0.17678, 0.18809, 0.10465])

titre_CHOB = np.array([0.00249, 0.01215, 0.02465, 0.05757, 0.13075,
                       0.1896, 0.4383, 0.58095, 0.7329, 0.90975,
                       1.4966, 1.85, 2.30745, 2.617, 2.662])
titre_CHOB_err = np.array([1.47e-3, 1.20e-4, 9.40e-4, 6.32e-3, 0.01379,
                           8.77e-3, 0.02630, 0.06314, 0.04045, 0.17854,
                           0.19714, 0.09334, 0.15635, 0.05798, 0.08485])

# --- Specific productivity (pg/cell/day) ---
qmab_CHOp = np.array([np.nan, 43.57, 29.71, 26.72, 73.07, 31.19,
                      29.65, 23.00, 43.61, 23.35, 29.95, 17.56,
                      -12.32, -3.04, -10.61])
qmab_CHOp_err = np.array([np.nan, 0.76, 4.38, 5.87, 11.06, 4.51,
                          12.59, 18.99, 15.37, 8.02, 19.91, 14.01,
                          5.86, 7.10, 11.25])

qmab_CHO = np.array([np.nan, 51.40, 27.10, 28.82, 31.93, 18.70,
                     36.77, 21.51, 12.96, 19.99, 32.76, 42.73,
                     22.48, 7.44, 0])
qmab_CHO_err = np.array([np.nan, 6.31, 9.47, 0.20, 8.77, 4.13,
                         9.35, 3.50, 4.57, 6.77, 5.39, 9.09,
                         22.45, 1.77, 12.10])

qmab_CHOB = np.array([np.nan, 33.63, 23.80, 41.12, 42.54, 17.06,
                      51.03, 21.34, 17.38, 18.06, 62.46, 44.72,
                      60.30, 41.03, 7.23])
qmab_CHOB_err = np.array([np.nan, 4.71, 3.30, 7.63, 15.04, 4.69,
                          3.54, 14.04, 12.31, 14.15, 1.29, 17.38,
                          8.05, 17.29, 23.86])

# --- Phase-specific qmAb (calculated averages) ---
# Exponential: days 0-8, Stationary: days 8-11, Death: days 11-14
# Values from paper text
phase_labels = ['Exponential\n(days 0–8)', 'Stationary\n(days 8–11)', 'Death\n(days 11–14)']
phase_qmab_CHO  = np.array([25.0, 25.5, 12.3])
phase_qmab_CHOp = np.array([31.7, 25.7, np.nan])  # CHO' death phase not reported — calculate
phase_qmab_CHOB = np.array([25.3, 35.7, 38.2])

phase_err_CHO  = np.array([8.8, 6.0, 0.0])
phase_err_CHOp = np.array([8.5, 4.1, np.nan])
phase_err_CHOB = np.array([3.4, 3.4, 6.3])

# Calculate CHO' death phase from raw data: days 12-14 qmab
# qmab_CHOp days 12,13,14 = -12.32, -3.04, -10.61 → mean ≈ -8.66
phase_qmab_CHOp[2] = np.mean([-12.32, -3.04, -10.61])  # ≈ -8.66
phase_err_CHOp[2] = np.std([-12.32, -3.04, -10.61])


# ══════════════════════════════════════════════════════════════════════
# FIGURE 5: GROWTH PROFILES
# ══════════════════════════════════════════════════════════════════════
fig5, ax5 = plt.subplots(figsize=(6, 4))

ax5.errorbar(days, vcd_CHO, yerr=vcd_CHO_err, color=c_CHO,
             linewidth=1.3, capsize=3, capthick=0.8, elinewidth=0.8,
             label='CHO')
ax5.errorbar(days, vcd_CHOp, yerr=vcd_CHOp_err, color=c_CHOp,
             linewidth=1.3, capsize=3, capthick=0.8, elinewidth=0.8,
             label="CHO\u2032")
ax5.errorbar(days, vcd_CHOB, yerr=vcd_CHOB_err, color=c_CHOB,
             linewidth=1.3, capsize=3, capthick=0.8, elinewidth=0.8,
             label='CHO-B')

# Temperature shift line
ax5.axvline(x=4, color='0.6', linewidth=0.8, linestyle='--', zorder=1)
ax5.annotate('32.5°C', xy=(4.2, 11.5), fontsize=8, color='0.4',
             fontstyle='italic')
ax5.annotate('36.5°C', xy=(2.5, 11.5), fontsize=8, color='0.4',
             fontstyle='italic')

ax5.set_xlabel('Days')
ax5.set_ylabel('Viable cell density (10⁶ cells/mL)')
ax5.set_xticks(np.arange(0, 15, 2))
ax5.set_xlim(-0.5, 14.5)
ax5.set_ylim(0, 13)
ax5.legend(loc='upper left')

plt.tight_layout()
# superseded working figure, no longer written
plt.show()
plt.close(fig5)
print("Figure 5 saved.")


# ══════════════════════════════════════════════════════════════════════
# FIGURE 6: ANTIBODY PRODUCTION (4-panel)
# ══════════════════════════════════════════════════════════════════════
fig6 = plt.figure(figsize=(10, 7.5))

# Layout: 2x2 grid
ax_titre = fig6.add_subplot(2, 2, 1)     # Top left: titre over time
ax_final = fig6.add_subplot(2, 2, 2)     # Top right: final titre bar
ax_qmab  = fig6.add_subplot(2, 2, 3)     # Bottom left: daily qmab
ax_phase = fig6.add_subplot(2, 2, 4)     # Bottom right: phase-specific qmab

# --- A: Titre over time ---
ax_titre.errorbar(days, titre_CHO, yerr=titre_CHO_err, color=c_CHO,
                  linewidth=1.3, capsize=2.5, capthick=0.7, elinewidth=0.7,
                  label='CHO')
ax_titre.errorbar(days, titre_CHOp, yerr=titre_CHOp_err, color=c_CHOp,
                  linewidth=1.3, capsize=2.5, capthick=0.7, elinewidth=0.7,
                  label="CHO\u2032")
ax_titre.errorbar(days, titre_CHOB, yerr=titre_CHOB_err, color=c_CHOB,
                  linewidth=1.3, capsize=2.5, capthick=0.7, elinewidth=0.7,
                  label='CHO-B')

ax_titre.set_xlabel('Days')
ax_titre.set_ylabel('IgG titre (g/L)')
ax_titre.set_xticks(np.arange(0, 15, 2))
ax_titre.set_xlim(-0.5, 14.5)
ax_titre.legend(loc='upper left')
ax_titre.text(-0.15, 1.05, 'A', transform=ax_titre.transAxes,
              fontsize=14, fontweight='bold', va='top')

# --- B: Final titre bar chart ---
final_titres = [titre_CHO[-1], titre_CHOp[-1], titre_CHOB[-1]]
final_errs   = [titre_CHO_err[-1], titre_CHOp_err[-1], titre_CHOB_err[-1]]
bar_colors   = [c_CHO, c_CHOp, c_CHOB]
bar_labels   = ['CHO', "CHO\u2032", 'CHO-B']
x_bar = np.arange(3)

bars = ax_final.bar(x_bar, final_titres, yerr=final_errs, width=0.6,
                    color=bar_colors, edgecolor='white', linewidth=0.5,
                    capsize=4, error_kw={'elinewidth': 0.8, 'capthick': 0.8})
ax_final.set_xticks(x_bar)
ax_final.set_xticklabels(bar_labels)
ax_final.set_ylabel('Final IgG titre (g/L)')
ax_final.set_ylim(0, 3.2)

# Significance brackets
def add_bracket(ax, x1, x2, y, h, text):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=0.8, color='0.3')
    ax.text((x1+x2)/2, y+h+0.02, text, ha='center', va='bottom',
            fontsize=8, color='0.3')

add_bracket(ax_final, 0, 2, 2.85, 0.05, '*')
add_bracket(ax_final, 1, 2, 2.65, 0.05, '**')

ax_final.text(-0.15, 1.05, 'B', transform=ax_final.transAxes,
              fontsize=14, fontweight='bold', va='top')

# --- C: Daily specific productivity ---
days_q = days[1:]  # skip day 0 (no qmab)
ax_qmab.errorbar(days_q, qmab_CHO[1:], yerr=qmab_CHO_err[1:], color=c_CHO,
                 linewidth=1.3, capsize=2.5, capthick=0.7, elinewidth=0.7,
                 label='CHO')
ax_qmab.errorbar(days_q, qmab_CHOp[1:], yerr=qmab_CHOp_err[1:], color=c_CHOp,
                 linewidth=1.3, capsize=2.5, capthick=0.7, elinewidth=0.7,
                 label="CHO\u2032")
ax_qmab.errorbar(days_q, qmab_CHOB[1:], yerr=qmab_CHOB_err[1:], color=c_CHOB,
                 linewidth=1.3, capsize=2.5, capthick=0.7, elinewidth=0.7,
                 label='CHO-B')

ax_qmab.axhline(0, color='0.6', linewidth=0.5, linestyle='-', zorder=1)
ax_qmab.set_xlabel('Days')
ax_qmab.set_ylabel('Specific productivity\n(pg/cell/day)')
ax_qmab.set_xticks(np.arange(0, 15, 2))
ax_qmab.set_xlim(-0.5, 14.5)
ax_qmab.legend(loc='upper left')
ax_qmab.text(-0.15, 1.05, 'C', transform=ax_qmab.transAxes,
             fontsize=14, fontweight='bold', va='top')

# --- D: Phase-specific qmAb bar chart ---
x_phase = np.arange(3)
bar_w = 0.22
offset = [-bar_w, 0, bar_w]

for i, (vals, errs, col, lbl) in enumerate([
    (phase_qmab_CHO, phase_err_CHO, c_CHO, 'CHO'),
    (phase_qmab_CHOp, phase_err_CHOp, c_CHOp, "CHO\u2032"),
    (phase_qmab_CHOB, phase_err_CHOB, c_CHOB, 'CHO-B'),
]):
    ax_phase.bar(x_phase + offset[i], vals, bar_w, yerr=errs,
                 color=col, edgecolor='white', linewidth=0.5,
                 capsize=3, error_kw={'elinewidth': 0.8, 'capthick': 0.8},
                 label=lbl)

ax_phase.set_xticks(x_phase)
ax_phase.set_xticklabels(phase_labels, fontsize=8)
ax_phase.set_ylabel('Average q$_{mAb}$\n(pg/cell/day)')
ax_phase.axhline(0, color='0.6', linewidth=0.5, linestyle='-', zorder=1)
ax_phase.legend(loc='upper left', fontsize=8)

# Significance markers on phase bars
# Stationary: CHO-B vs CHO p<0.05
add_bracket(ax_phase, 1 - bar_w, 1 + bar_w, 40, 1.5, '*')
# Death: CHO-B vs CHO p<0.01
add_bracket(ax_phase, 2 - bar_w, 2 + bar_w, 46, 1.5, '**')

ax_phase.text(-0.15, 1.05, 'D', transform=ax_phase.transAxes,
              fontsize=14, fontweight='bold', va='top')

plt.tight_layout()
# superseded working figure, no longer written

In [ ]:
"""Figure 4 (rebuilt): integrated process performance, six panels, 3x2.

Consolidates panels currently spread across Figures 4-6 so a reader can tell
whether the titre gain came from more cells, a longer culture, or higher
per-cell productivity.

A VCD          B viability        (row 1)
C IVCD         D IgG titre        (row 2)
E qmAb by phase  F leucine flux   (row 3)

A/C and B/D share a daily x-axis and are aligned vertically. The day-4
temperature shift is marked on every time-course panel.

Time courses come from Experiments/BCAT1_KO Fed-Batch.xlsx (FinalResults);
phase-averaged qmAb and the leucine flux trace come from Figures.ipynb.
"""

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import openpyxl

HERE = Path.cwd()
XLSX = HERE.parent / "data" / "BCAT1_KO Fed-Batch.xlsx"

C_CHO, C_CHOP, C_CHOB = "#2166AC", "#E08214", "#1A9850"
LINES = [("CHO", 15, C_CHO), ("CHO\u2032", 2, C_CHOP), ("CHO-B", 28, C_CHOB)]
SHIFT_DAY = 4

mpl.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "serif", "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9, "axes.labelsize": 9.5, "xtick.labelsize": 8,
    "ytick.labelsize": 8, "legend.fontsize": 8, "legend.frameon": False,
    "figure.dpi": 300, "savefig.dpi": 300,
    "axes.linewidth": .8, "xtick.major.width": .8, "ytick.major.width": .8,
    "xtick.direction": "in", "ytick.direction": "in",
    "axes.spines.top": False, "axes.spines.right": False,
})

# --- time courses from the workbook -----------------------------------------
# column offsets from each cell line's block start, per the FinalResults header
DAY, VCD, VCD_SD, VIAB, IVCD, IVCD_SD, TITRE, TITRE_SD = 0, 2, 3, 4, 5, 6, 8, 9

ws = openpyxl.load_workbook(XLSX, data_only=True, read_only=True)["FinalResults"]
ROWS = [r for r in ws.iter_rows(min_row=4, max_row=18, values_only=True)]


def series(base, offset):
    out = []
    for row in ROWS:
        try:
            out.append(float(row[base + offset]))
        except (TypeError, ValueError):
            out.append(np.nan)
    return np.array(out)


# viability is only averaged in FinalResults; the two replicates live in
# `Results`, in three blocks stacked at rows 2 / 24 / 46 (rep 1 col 6, rep 2 col 17)
VIAB_BLOCK = {"CHO′": 5, "CHO": 27, "CHO-B": 49}
RES = openpyxl.load_workbook(XLSX, data_only=True, read_only=True)["Results"]
RES_ROWS = [r for r in RES.iter_rows(min_row=1, max_row=63, max_col=26,
                                     values_only=True)]


def viability_replicates(first_row):
    """Per-replicate viability recomputed from the primary counts, VCD/(VCD+DCD).

    The stored viability column is used only as a cross-check: it agrees
    everywhere except CHO' day 4 replicate 2, which records 0% alongside
    3.2e6 viable cells and is a transcription error.
    """
    reps = []
    for row in RES_ROWS[first_row - 1:first_row - 1 + 15]:
        pair = []
        for vcd_c, dcd_c in ((4, 5), (15, 16)):
            try:
                vcd, dcd = float(row[vcd_c]), float(row[dcd_c])
                pair.append(vcd / (vcd + dcd) if vcd + dcd > 0 else np.nan)
            except (TypeError, ValueError):
                pair.append(np.nan)
        reps.append(pair)
    arr = 100 * np.asarray(reps)
    return arr.mean(axis=1), arr.std(axis=1, ddof=1)


days = series(15, DAY)
data = {name: {k: series(base, off) for k, off in
               (("vcd", VCD), ("vcd_sd", VCD_SD), ("viab", VIAB),
                ("ivcd", IVCD), ("ivcd_sd", IVCD_SD),
                ("titre", TITRE), ("titre_sd", TITRE_SD))}
        for name, base, _ in LINES}

for name, _, _ in LINES:
    mean, sd = viability_replicates(VIAB_BLOCK[name])
    data[name]["viab_mean"], data[name]["viab_sd"] = mean, sd

# guard the block mapping against the values published in Figures.ipynb
assert np.allclose(data["CHO\u2032"]["vcd"][:4], [0.2, 0.52, 1.12, 1.15])
assert np.allclose(data["CHO"]["vcd"][:4], [0.2, 0.378, 0.793, 1.48])
assert np.allclose(data["CHO-B"]["titre"][-1], 2.662, atol=1e-3)
# UNRESOLVED: the two sheets carry different viability series. `Results` is
# internally consistent (its stored column equals VCD/(VCD+DCD) everywhere bar
# the one bad cell), but its replicate means do not reproduce the `FinalResults`
# viability. Panel B is drawn from `Results`; flip VIABILITY_SOURCE to compare.
VIABILITY_SOURCE = "Results"        # or "FinalResults" (no error bars available there)
print("viability: max |Results mean - FinalResults| per line, percentage points")
for name, _, _ in LINES:
    gap = np.nanmax(np.abs(data[name]["viab_mean"] - 100 * data[name]["viab"]))
    print(f"   {name:6s} {gap:6.1f}")

# --- phase-averaged qmAb and leucine flux, from Figures.ipynb ---------------
PHASES = ["Exponential\n(days 0\u20138)", "Stationary\n(days 8\u201311)",
          "Death\n(days 11\u201314)"]
PHASE_Q = {"CHO": ([26.0, 34.4, 21.9], [4.0, 2.6, 5.7]),
           "CHO\u2032": ([35.3, 29.3, 6.0], [4.1, 5.6, 1.4]),
           "CHO-B": ([30.3, 46.2, 56.6], [1.3, 1.0, 11.8])}

INTERVALS = ["5\u20136", "6\u20137", "7\u20138", "8\u20139", "9\u201310",
             "10\u201311", "11\u201312", "12\u201313", "13\u201314"]
FLUX = {
    "CHO": ([-.0025, -.0101, -.0021, -.0068, -.0009, -.0122, -.0018, -.0109, -.0031],
            [.0014, .0052, .0017, .0038, .0017, .0050, .0012, .0074, .0036]),
    "CHO\u2032": ([-.0051, -.0111, -.0014, .0008, -.0070, -.0099, .0025, -.0146, -.0027],
                  [.0069, .0020, .0020, .0045, .0081, .0049, .0008, .0015, .0022]),
    "CHO-B": ([-.0002, -.0148, -.0006, -.0092, -.0031, -.0101, -.0041, -.0092, -.0018],
              [.0063, .0018, .0006, .0006, .0024, .0012, .0021, .0031, .0017]),
}
CDW_NG, HOURS = 0.315, 24.0        # ng/cell; h/day -> pmol cell-1 day-1


def shift_marker(ax, label_y, first=False):
    ax.axvline(SHIFT_DAY, color="0.6", lw=.8, ls="--", zorder=1)
    if first:
        ax.annotate("36.5\u00b0C", xy=(SHIFT_DAY - 0.3, label_y), fontsize=7,
                    color="0.4", style="italic", ha="right")
        ax.annotate("32.5\u00b0C", xy=(SHIFT_DAY + 0.3, label_y), fontsize=7,
                    color="0.4", style="italic", ha="left")


def panel_letter(ax, letter):
    ax.text(-0.17, 1.06, letter, transform=ax.transAxes, fontsize=12,
            fontweight="bold", va="top")


def errline(ax, x, y, e, colour, label):
    ax.errorbar(x, y, yerr=e, color=colour, lw=1.3, capsize=2.5,
                capthick=.7, elinewidth=.7, label=label, zorder=3)


fig, axes = plt.subplots(3, 2, figsize=(7.1, 8.5))
fig.subplots_adjust(left=.095, right=.985, top=.965, bottom=.065,
                    hspace=.34, wspace=.28)
(axA, axB), (axC, axD), (axE, axF) = axes

# A -- VCD
for name, _, colour in LINES:
    errline(axA, days, data[name]["vcd"], data[name]["vcd_sd"], colour, name)
shift_marker(axA, 12.0, first=True)
axA.set_ylabel("VCD (10$^6$ cells mL$^{-1}$)")
axA.set_ylim(0, 13.2); axA.legend(loc="lower right")
panel_letter(axA, "A")

# B -- viability. Full 0-100 scale: on a 70-105 axis the replicate scatter
# reads as a sawtooth that the error bars show is not there.
for name, _, colour in LINES:
    if VIABILITY_SOURCE == "Results":
        errline(axB, days, data[name]["viab_mean"], data[name]["viab_sd"],
                colour, name)
    else:
        axB.plot(days, 100 * data[name]["viab"], color=colour, lw=1.3, label=name)
shift_marker(axB, 96)
axB.set_ylabel("Viability (%)")
axB.set_ylim(0, 104); axB.set_yticks(np.arange(0, 101, 20))
panel_letter(axB, "B")

# C -- IVCD, the load-bearing panel
for name, _, colour in LINES:
    errline(axC, days, data[name]["ivcd"], data[name]["ivcd_sd"], colour, name)
shift_marker(axC, 2320)
axC.set_ylabel("IVCD (10$^6$ cells mL$^{-1}$ h)")
axC.set_ylim(0, 2500)
for name, _, colour in LINES:                       # final values, spelled out
    axC.annotate(f"{data[name]['ivcd'][-1]:.0f}", xy=(14, data[name]["ivcd"][-1]),
                 xytext=(3, 0), textcoords="offset points", fontsize=7.5,
                 color=colour, va="center")
axC.set_xlabel("Culture time (days)")
panel_letter(axC, "C")

# D -- titre
for name, _, colour in LINES:
    errline(axD, days, data[name]["titre"], data[name]["titre_sd"], colour, name)
shift_marker(axD, 2.9)
axD.set_ylabel("IgG titre (g L$^{-1}$)")
axD.set_ylim(0, 3.1)
# Peak titre, not the day-14 endpoint: the text compares peaks, and
# labelling the endpoints invited the reader to compute a different figure.
for name, _, colour in LINES:
    k = int(np.nanargmax(data[name]["titre"]))
    axD.plot(days[k], data[name]["titre"][k], "o", color=colour, ms=4.5,
             mfc="white", mew=1.2, zorder=5)
    off = {"CHO": (17, 7), "CHO′": (-17, 9), "CHO-B": (2, 13)}[name]
    axD.annotate(f"{data[name]['titre'][k]:.2f}",
                 xy=(days[k], data[name]["titre"][k]),
                 xytext=off, textcoords="offset points", fontsize=7.5,
                 color=colour, ha="center", va="center")
axD.set_xlabel("Culture time (days)")
panel_letter(axD, "D")

for ax in (axA, axB, axC, axD):
    ax.set_xlim(-0.5, 15.4); ax.set_xticks(np.arange(0, 15, 2))
for ax in (axA, axB):
    ax.set_xlabel("Culture time (days)")

# E -- phase-averaged qmAb
x = np.arange(3); width = .24
for i, (name, _, colour) in enumerate(LINES):
    vals, errs = PHASE_Q[name]
    axE.bar(x + (i - 1) * width, vals, width, yerr=errs, color=colour,
            edgecolor="white", lw=.5, capsize=3,
            error_kw={"elinewidth": .8, "capthick": .8}, label=name)
axE.axhline(0, color="0.6", lw=.5, zorder=1)
axE.set_xticks(x); axE.set_xticklabels(PHASES, fontsize=7.5)
axE.set_ylabel("Average q$_{mAb}$ (pg cell$^{-1}$ day$^{-1}$)")
axE.set_ylim(-4, 74)
axE.legend(loc="upper left", ncol=3, columnspacing=1.1)
panel_letter(axE, "E")

# F -- specific leucine flux
xf = np.arange(len(INTERVALS))
for name, _, colour in LINES:
    flux, err = FLUX[name]
    axF.errorbar(xf, flux, yerr=err, color=colour, lw=1.2, capsize=3,
                 capthick=.8, elinewidth=.8, label=name)
axF.axhline(0, color="0.6", lw=.5, zorder=1)
# Forced minimum leucine uptake imposed by the D2 and D4 constraint files.
# Uptake is negative here, so points ABOVE the band did not reach the minimum.
FORCED_MIN = {"D2": -0.008593, "D4": -0.010148}
axF.axhspan(FORCED_MIN["D4"], FORCED_MIN["D2"], color="0.88", zorder=0, lw=0)
axF.annotate('FBA forced minimum uptake (D2–D4)', xy=(0.015, 0.035),
             xycoords="axes fraction", fontsize=5.8, color="0.35",
             va="bottom", ha="left")
axF.set_xticks(xf); axF.set_xticklabels(INTERVALS, rotation=45, ha="right")
axF.set_xlabel("Culture interval (days)")
axF.set_ylabel("Specific leucine uptake rate\n(mmol gDCW$^{-1}$ h$^{-1}$)")
panel_letter(axF, "F")

save(fig, 'Fig4_Fed-batch culture performance of parental, electroporation-control and BCAT1-knockdown cell lines')

# --- alternative panel F: cumulative leucine consumed per cell --------------
fig2, ax = plt.subplots(figsize=(3.5, 2.9))
fig2.subplots_adjust(left=.20, right=.97, top=.95, bottom=.20)
for name, _, colour in LINES:
    flux, err = FLUX[name]
    per_cell = -np.asarray(flux) * CDW_NG * HOURS      # pmol cell-1 day-1
    cum = np.cumsum(per_cell)
    cum_sd = np.sqrt(np.cumsum((np.asarray(err) * CDW_NG * HOURS) ** 2))
    ax.errorbar(xf, cum, yerr=cum_sd, color=colour, lw=1.3, capsize=2.5,
                capthick=.7, elinewidth=.7, label=name)
ax.axhline(0, color="0.6", lw=.5, zorder=1)
ax.set_xticks(xf); ax.set_xticklabels(INTERVALS, rotation=45, ha="right")
ax.set_xlabel("Culture interval (days)")
ax.set_ylabel("Cumulative leucine consumed\n(pmol cell$^{-1}$)")
ax.legend(loc="upper left")
save(fig2, 'FigS3_Cumulative leucine consumed per cell over the fed-batch')

print("final IVCD :", {n: round(data[n]["ivcd"][-1], 1) for n, _, _ in LINES})
print("final titre:", {n: round(data[n]["titre"][-1], 3) for n, _, _ in LINES})
for name, _, _ in LINES:
    flux, _e = FLUX[name]
    cum = np.cumsum(-np.asarray(flux) * CDW_NG * HOURS)[-1]
    print(f"  {name:6s} cumulative leucine {cum:7.4f} pmol/cell   "
          f"titre/leucine {data[name]['titre'][-1] / cum:6.3f}")
print("written figure4_process_performance.pdf/.png and "
      "figureS_cumulative_leucine.pdf/.png")

plt.show()
